# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset of mass spectrometry analyses using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

First, we'll set up the dataset URL and load the metadata. This step will give us information about the dataset, including its title, description, and available record sets.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
ds = mlc.Dataset(croissant_url)
metadata = ds.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Let's review the available record sets, their fields, and the `@id` for each, using the Croissant introspection API.

Each record set may represent a table or logical collection of records, with fields mapped to columns.

In [ ]:
# List record sets by @id and get fields for each
print("Available Record Sets (by @id):\n--------------------------------")
for record_set in metadata.record_sets:
    print(f"Record Set @id: {record_set.id}, name: {record_set.name}")
    print("  Fields and their @ids:")
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
    print()
# Save a list of record set @ids for later use
record_set_ids = [rs.id for rs in metadata.record_sets]

## 3. Data Extraction
We now extract data from a specific record set into a DataFrame for analysis. Pick a record set `@id` from the overview above. Typically, the main table contains records about proteins, their abundance, description, and related numeric fields. We'll load **all record sets** into separate DataFrames by their `@id`.

In [ ]:
# Build DataFrames for all record sets found
dataframes = {}
for recset_id in record_set_ids:
    records = list(ds.records(record_set=recset_id))
    dataframes[recset_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records in record set @id: {recset_id}")

# For exploration, pick the first record set (update this selection if another is the main one)
target_rs_id = record_set_ids[0] if record_set_ids else None
if target_rs_id and not dataframes[target_rs_id].empty:
    print("\nColumns in the main record set:")
    print(dataframes[target_rs_id].columns.tolist())
    display(dataframes[target_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
We'll apply simple processing steps:

- Select a numeric field for example analysis (e.g., molecular weight, coverage, or peptide count).
- Filter records based on a threshold.
- Normalize a numeric field.
- Optionally, group by a category field if available (such as a protein attribute).

Modify the selected fields based on the actual fields listed in the previous cell.

In [ ]:
# Replace below with field ids from your chosen record set as appropriate!
rs_id = target_rs_id  # Example record set selected
df = dataframes[rs_id]

# Try to auto-detect numeric fields
numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print(f"Numeric field candidates in this record set: {numeric_candidates}")
if not numeric_candidates:
    # Try to convert likely numeric columns
    possible_numeric = [col for col in df.columns if any(keyword in col.lower() for keyword in ["weight", "mw", "count", "intensity", "coverage", "abundance"])]
    for col in possible_numeric:
        try:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        except:
            pass
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    print(f"After conversion, numeric field candidates: {numeric_candidates}")

# Choose a numeric field for demonstration
if numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    print("No numeric fields detected; please adjust numeric_field manually.")
    numeric_field = df.columns[0]  # Fallback

threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
filtered_df = df[df[numeric_field] > threshold]
print(f"\nFiltered records with {numeric_field} > {threshold:.3f}:")
display(filtered_df.head())

filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

# Try to find a likely grouping field (e.g., 'description' or one with < 10 unique values)
group_field = None
for col in filtered_df.columns:
    if 1 < filtered_df[col].nunique() < 10 and not pd.api.types.is_numeric_dtype(filtered_df[col]):
        group_field = col
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"\nGrouped data by {group_field} (mean of each group):")
    display(grouped_df.head())
else:
    print("\nNo suitable group field detected for grouping.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field for the filtered records and the relationship to the grouping variable (if present).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Histogram of the normalized numeric field
plt.figure(figsize=(7,4))
sns.histplot(filtered_df[numeric_field + '_normalized'].dropna(), bins=30, kde=True)
plt.title(f'Distribution of {numeric_field} (normalized)')
plt.xlabel(f'{numeric_field} (normalized)')
plt.ylabel('Count')
plt.show()

# Boxplot by group if group field is present
if group_field:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field, y=numeric_field + '_normalized', data=filtered_df)
    plt.title(f'Normalized {numeric_field} by {group_field}')
    plt.show()

## 6. Conclusion
In this notebook we:

- Loaded the FAIR^2 mass spectrometry dataset via its Croissant schema.
- Explored the available record sets, fields, and their IDs.
- Loaded all record sets into DataFrames for Python analysis referencing all entities by their `@id`s.
- Demonstrated basic filtering, normalization, and grouping on one numeric field.
- Visualized distributions for the filtered and normalized data.

**You are now ready to perform deeper data analysis on this and other Croissant-compliant open datasets!**